# 介紹
使用 pyautogui 來完成小恐龍的遊戲機器人。

小恐龍是在 chrome 中使用 chrome://dino 啟動。

邏輯：

不斷監視畫面
讀取某個位置的顏色
若此位置上出現 (83, 83, 83) 的顏色
跳起來

In [1]:
import time
import pyautogui as pyg

pyg.PAUSE = 0.0001
pyg.sleep(2)
pyg.press('space')

start_time = time.time()

while True:
    current_time = time.time() - start_time

    # 動態拉遠偵測距離（隨著時間變快拉長前方偵測）
    extra_lookahead = min(int(current_time / 4), 40)
    scan_x = 220+ extra_lookahead

    # 1. 檢查背景顏色判斷日夜模式
    bg_sample = pyg.screenshot(region=(scan_x, 340, 10, 10))
    bg_color = bg_sample.getpixel((5, 5))
    is_dark_mode = bg_color[0] < 100

    # 2. 截取前方障礙物偵測區 (Y 軸 280 到 450)
    im = pyg.screenshot(region=(scan_x, 280, 60, 170))
    width, height = im.size

    obstacle_ys = []

    for x in range(width):
        for y in range(height):
            c = im.getpixel((x, y))

            if is_dark_mode:
                is_obstacle = c[0] > 150 and c[1] > 150 and c[2] > 150
            else:
                is_obstacle = c[0] < 120 and c[1] < 120 and c[2] < 120

            if is_obstacle:
                obstacle_ys.append(y)

    # 3. 精確判斷：直走、低頭蹲下、最高跳躍
    if obstacle_ys:
        min_y = min(obstacle_ys)  # 障礙物最頂端像素
        max_y = max(obstacle_ys)  # 障礙物最底端像素

        abs_min_y = 280 + min_y    # 換算螢幕絕對 Y 座標
        abs_max_y = 280 + max_y

        # 【情況 1】高空翼龍：整體飛在頭頂上方 -> 直走放過
        if abs_max_y < 330:
            pass

        # 【情況 2】中空翼龍：強制作低頭蹲下
        elif abs_min_y < 395:
            pyg.keyDown('down')
            time.sleep(0.18)
            pyg.keyUp('down')

        # 【情況 3】地表障礙物 -> 觸發最高高度跳躍
        else:
            pyg.keyDown('space')
            time.sleep(0.18)  # 持續按住 space 達到最高跳躍
            pyg.keyUp('space')
            time.sleep(0.1)   # 等待恐龍完整完成滯空

KeyboardInterrupt: 